# Figuras del anexo · 21 a 40

Complemento del notebook 10. Aquellas veinte son las que **argumentan** el capítulo; éstas
veinte son las que **demuestran el trabajo hecho**: las representaciones clásicas del análisis
exploratorio —mapas de calor, dendrograma, PCA, ACF/PACF, descomposición estacional, Q-Q,
violines— que en el cuerpo no caben pero en un anexo sí, y que un tribunal espera ver.

## Dos decisiones de diseño

**Estilo unificado.** Todas comparten paleta, tipografía, tamaños y tratamiento de ejes, que se
declaran una vez en el bloque 0. Veinte figuras con seis estilos distintos parecen recortadas de
sitios distintos; con uno solo, parecen un trabajo.

**Láminas compuestas.** Al final hay cinco *láminas* que agrupan cuatro paneles cada una en una
sola página. Es la forma de meter mucho análisis en poco espacio sin que parezca un vertedero de
gráficos: cada lámina tiene un título y un pie que la lee como conjunto.

## Índice

| Grupo | Figuras | Contenido |
|---|---|---|
| **A** | 21-25 | Mapas de calor: correlación, mes×hora, ausencias, calendario, semanal |
| **B** | 26-29 | Estructura multivariante: dendrograma, PCA, cargas, V de Cramér |
| **C** | 30-33 | Distribuciones: histogramas, cajas por familia, Q-Q, curva de duración |
| **D** | 34-37 | Estructura temporal: ACF/PACF, descomposición, densidades, violines |
| **E** | 38-40 | Estabilidad y evaluación: correlación por año, dispersión, coste |
| **Láminas** | L1-L5 | Cinco composiciones de cuatro paneles para presentación compacta |

---
## 0 · Estilo unificado y carga

El bloque de estilo es el mismo del notebook 10, ampliado con una paleta declarada. Si se cambia
aquí, cambian las cuarenta figuras a la vez — que es justamente el motivo de tenerlo en un solo
sitio.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

%matplotlib inline
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*PeriodArray.*")

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SALIDA = RAIZ / "docs" / "figuras"
TZ = "Europe/Madrid"

APAGON = (pd.Timestamp("2025-04-28", tz="UTC"), pd.Timestamp("2025-05-06 23:00", tz="UTC"))
EXCEPCION = (pd.Timestamp("2022-06-15", tz="UTC"), pd.Timestamp("2023-12-31 23:00", tz="UTC"))

# --- Paleta única para las 40 figuras -------------------------------------
PALETA = {
    "principal": "#1f4e79",   # azul oscuro: series principales
    "secundario": "#2b7bba",  # azul medio
    "acento": "#f5a623",      # ámbar: contrastes y segundas series
    "alerta": "#c0392b",      # rojo: con fuga, avisos, horas caras
    "ok": "#2ca02c",          # verde: sin fuga, utilizable
    "neutro": "#7f7f7f",
    "suave": "#d9d9d9",
}
COLOR_FUGA = {"SIN FUGA": PALETA["ok"], "DESFASE D-2": PALETA["acento"],
              "CON FUGA": PALETA["alerta"], "CONDICIONAL": PALETA["neutro"],
              "TARGET": "#000000"}
CMAP_DIV = "RdBu_r"      # divergente: correlaciones, desviaciones con signo
CMAP_SEQ = "viridis"     # secuencial: intensidades sin signo
CMAP_CAL = "RdYlBu_r"    # calendario y rangos de precio

PIES = []

def estilo():
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
        "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
        "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "figure.facecolor": "white", "axes.prop_cycle":
            plt.cycler(color=[PALETA["principal"], PALETA["acento"], PALETA["ok"],
                              PALETA["alerta"], PALETA["secundario"], PALETA["neutro"]]),
    })

def guardar(fig, n, nombre, pie, fmt="png", cerrar=True):
    '''Guarda, registra el pie y cierra la figura: con 40 abiertas matplotlib avisa.'''
    SALIDA.mkdir(parents=True, exist_ok=True)
    etq = f"fig{n:02d}" if isinstance(n, int) else n
    ruta = SALIDA / f"{etq}_{nombre}.{fmt}"
    fig.savefig(ruta)
    PIES.append((etq, ruta.name, pie))
    print(f"  {etq}  {ruta.name}")
    if cerrar:
        plt.show()
        plt.close(fig)

def pick(d, *c):
    for x in c:
        if x is not None and x in d.columns:
            return x
    return None

def sin_rejilla(ax):
    '''Los mapas de calor no llevan rejilla: se superpone a las celdas.'''
    ax.grid(False)
    return ax

In [ ]:
df = pd.read_parquet(RAIZ / "data" / "bronze" / "bronze_unificado.parquet")
df["ts_utc"] = pd.to_datetime(df["ts_utc"], utc=True)
assert df["ts_utc"].duplicated().sum() == 0, "parquet duplicado: regenerar"

df["ts_local"] = df["ts_utc"].dt.tz_convert(TZ)
df["hora"] = df["ts_local"].dt.hour
df["dia"] = df["ts_local"].dt.date
df["anio"] = df["ts_local"].dt.year
df["mes"] = df["ts_local"].dt.month
df["dow"] = df["ts_local"].dt.dayofweek
df["mes_p"] = df["ts_local"].dt.to_period("M")
df["regimen"] = np.select(
    [df["ts_utc"].between(*APAGON), df["ts_utc"].between(*EXCEPCION)],
    ["apagón", "excepción ibérica"], default="normal")

PRECIO = pick(df, "spot_es_esios", "spot_es_omie", "spot_es_entsoe")
if PRECIO is None:
    raise SystemExit("Falta spot_price en el bronce.")

CAND = {k: v for k, v in {
    "demanda prevista": (pick(df, "forecast_demanda_mercado_prev_mw"), "SIN FUGA"),
    "eólica prevista":  (pick(df, "forecast_gen_wind_prev_mw"), "SIN FUGA"),
    "solar prevista":   (pick(df, "forecast_gen_solar_pv_prev_mw"), "SIN FUGA"),
    "gas":              (pick(df, "tp_ttf_cierre", "commodities_gas_ttf_m1"), "DESFASE D-2"),
    "CO2":              (pick(df, "tp_eua_cierre", "commodities_co2_eua_dec"), "DESFASE D-2"),
    "demanda real":     (pick(df, "load_inter_entsoe_load"), "CON FUGA"),
    "eólica real":      (pick(df, "entsoe_wind_mw"), "CON FUGA"),
    "solar FV real":    (pick(df, "calc_solar_fv_mw"), "CON FUGA"),
    "hidráulica":       (pick(df, "calc_hydro_dispatch_mw"), "CON FUGA"),
    "ciclo combinado":  (pick(df, "esios_gen_ree_gccgas_mw"), "CON FUGA"),
    "temperatura":      (pick(df, "era5_t2m_mean"), "CON FUGA"),
    "viento 100m":      (pick(df, "era5_wind100_mean"), "CON FUGA"),
    "radiación":        (pick(df, "era5_ssrd_mean"), "CON FUGA"),
    "precio ES":        (PRECIO, "TARGET"),
}.items() if v[0]}

estilo()
print(f"{len(df):,} horas · target: {PRECIO} · {len(CAND)} candidatas resueltas")

---
# Grupo A · Mapas de calor

## Figura 21 · Matriz de correlación de las candidatas

El mapa de calor clásico, con el coeficiente escrito en cada celda y la **etiqueta de fuga** en
las etiquetas de fila. La diagonal superior se enmascara: la matriz es simétrica y repetirla
gasta espacio sin añadir nada.

Se usa una escala divergente centrada en cero, porque el signo importa tanto como la magnitud.

In [ ]:
cols = {k: v[0] for k, v in CAND.items()}
S = df[list(cols.values())].rename(columns={v: k for k, v in cols.items()})
Cm = S.corr()

fig, ax = plt.subplots(figsize=(8.5, 7))
mask = np.triu(np.ones_like(Cm, dtype=bool))
im = ax.imshow(np.where(mask, np.nan, Cm.values), cmap=CMAP_DIV,
               norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))
ax.set_xticks(range(len(Cm)), Cm.columns, rotation=45, ha="right")
ax.set_yticks(range(len(Cm)), [f"{c}  [{CAND[c][1]}]" for c in Cm.columns], fontsize=7.5)
for et, c in zip(ax.get_yticklabels(), Cm.columns):
    et.set_color(COLOR_FUGA.get(CAND[c][1], "black"))
for i in range(len(Cm)):
    for j in range(i):
        v = Cm.iloc[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if abs(v) > 0.55 else "black")
ax.set_title("Correlación entre las variables candidatas")
sin_rejilla(ax); fig.colorbar(im, ax=ax, shrink=0.7, label="r de Pearson")
plt.tight_layout()
guardar(fig, 21, "matriz_correlacion_candidatas",
        "Matriz de correlación de las variables candidatas. El color de la etiqueta indica si "
        "el dato está disponible a las 12:00 de D: verde utilizable, rojo con fuga.")

## Figura 22 · Mapa de calor mes × hora del precio

La representación más compacta de la doble estacionalidad. En un solo cuadro se ve el ciclo
diario, el anual, y **cómo interactúan**: el valle solar de mediodía existe en verano y casi no
en invierno, algo que ninguna serie temporal deja ver de un vistazo.

In [ ]:
heat = df[df["regimen"] == "normal"].pivot_table(
    index="hora", columns="mes", values=PRECIO, aggfunc="mean")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
im = ax.imshow(heat.values, aspect="auto", origin="lower", cmap=CMAP_SEQ)
ax.set_xticks(range(12), ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_yticks(range(0, 24, 2), range(0, 24, 2))
ax.set_xlabel("Mes"); ax.set_ylabel("Hora local")
ax.set_title("Precio medio por mes y hora del día")
sin_rejilla(ax); fig.colorbar(im, ax=ax, label="EUR/MWh")
plt.tight_layout()
guardar(fig, 22, "heatmap_mes_hora",
        "Precio medio en cada combinación de mes y hora. La doble estacionalidad y su "
        "interacción en una sola imagen: el valle de mediodía se profundiza en los meses de "
        "mayor radiación.")

## Figura 23 · Mapa de ausencias

Adaptación de `patron_perdidos` de las prácticas: correlación de la **indicadora de ausencia**
de cada columna. Dos columnas con correlación 1 faltan siempre a la vez.

Es la prueba formal de algo que el notebook 02 afirmaba por inspección: los nulos de ERA5 son
**una sola causa** —el paso trihorario— y no nueve problemas distintos.

In [ ]:
con_nulos = [c for c in df.columns
             if df[c].isna().any() and df[c].dtype.kind in "fi"]
grupos = {
    "ERA5 (3h)": [c for c in con_nulos if c.startswith("era5_")],
    "Capacidad y commodities (diarias)": [c for c in con_nulos
                                          if c.startswith(("cap_", "commodities_"))],
}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (nombre, cs) in zip(axes, grupos.items()):
    if len(cs) < 2:
        ax.axis("off"); continue
    M = df[cs].isna().corr()
    im = ax.imshow(M.values, cmap=CMAP_DIV, vmin=-1, vmax=1)
    ax.set_xticks(range(len(cs)), [c[:26] for c in cs], rotation=90, fontsize=6)
    ax.set_yticks(range(len(cs)), [c[:26] for c in cs], fontsize=6)
    ax.set_title(nombre, fontsize=9)
    sin_rejilla(ax)
fig.colorbar(im, ax=axes, shrink=0.6, label="correlación de ausencias")
guardar(fig, 23, "patron_ausencias",
        "Correlación entre las indicadoras de ausencia. Un bloque uniforme significa una sola "
        "causa: los nulos de ERA5 son su paso trihorario, y los de las tablas diarias, el "
        "aterrizaje en la hora 00 local. No son huecos de datos.")

## Figura 24 · Calendario de precios

Un año por fila y un día por columna. Es la vista que mejor revela **episodios**: olas de calor,
temporales de viento, el apagón. Una serie temporal larga los aplasta; el calendario los aísla.

In [ ]:
diario = df.groupby("dia")[PRECIO].mean()
diario.index = pd.to_datetime(diario.index)
anios = sorted(diario.index.year.unique())

fig, ax = plt.subplots(figsize=(11, 0.55 * len(anios) + 1.4))
M = np.full((len(anios), 366), np.nan)
for i, a in enumerate(anios):
    s = diario[diario.index.year == a]
    M[i, s.index.dayofyear.values - 1] = s.values
im = ax.imshow(M, aspect="auto", cmap=CMAP_CAL, interpolation="nearest")
ax.set_yticks(range(len(anios)), anios)
inicios = [pd.Timestamp(f"2024-{m:02d}-01").dayofyear for m in range(1, 13)]
ax.set_xticks(inicios, ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_xlabel("Día del año")
ax.set_title("Precio medio diario · un año por fila")
sin_rejilla(ax); fig.colorbar(im, ax=ax, shrink=0.85, label="EUR/MWh")
plt.tight_layout()
guardar(fig, 24, "calendario_precio",
        "Calendario de precios medios diarios. Aísla episodios que una serie temporal larga "
        "aplasta: crisis de precios, olas de calor y el apagón ibérico se leen como manchas.")

## Figura 25 · Perfil semanal: día de la semana × hora

Separa el efecto calendario del efecto horario. Si el fin de semana tiene otro perfil —y no solo
otro nivel—, el modelo necesita interacción entre día y hora, no una simple variable de
laboralidad.

In [ ]:
DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
sem = df[df["regimen"] == "normal"].pivot_table(
    index="hora", columns="dow", values=PRECIO, aggfunc="mean")
sem.columns = DIAS

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(sem.values, aspect="auto", origin="lower", cmap=CMAP_SEQ)
axes[0].set_xticks(range(7), DIAS); axes[0].set_yticks(range(0, 24, 2), range(0, 24, 2))
axes[0].set_ylabel("Hora local"); axes[0].set_title("Precio medio por día y hora")
sin_rejilla(axes[0]); fig.colorbar(im, ax=axes[0], label="EUR/MWh")

sem.plot(ax=axes[1], lw=1.3)
axes[1].set_xlabel("Hora local"); axes[1].set_ylabel("EUR/MWh")
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_title("Los mismos perfiles superpuestos")
axes[1].legend(frameon=False, ncol=4, fontsize=7)
plt.tight_layout()
guardar(fig, 25, "perfil_semanal",
        "Perfil horario del precio según el día de la semana. Un fin de semana con forma "
        "distinta —y no sólo nivel distinto— exige interacción entre día y hora en el modelo.")

---
# Grupo B · Estructura multivariante

## Figura 26 · Dendrograma de agrupamiento

Enlace jerárquico sobre la distancia `1 − |r|`. Cada rama que se une por debajo de la línea roja
agrupa columnas que correlacionan 0,90 o más: **candidatas a que sobre alguna**.

En horizontal, que con muchas etiquetas es la única orientación legible. Se calcula sobre el
marco **diario**, donde todas las series comparten frecuencia y no hay pares indefinidos.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

num = [c for c in df.select_dtypes(include=[np.number]).columns
       if c not in ("hora", "anio", "mes", "dow", "hour_utc", "hour_local", "tensor_index")]
# Fuera las constantes: su correlación es indefinida y el árbol las coloca como si fueran
# las más distintas de todas, cuando en realidad no tienen información.
constantes = [c for c in num if df[c].nunique(dropna=True) <= 1]
num = [c for c in num if c not in constantes]
print(f"{len(constantes)} columnas constantes excluidas: {constantes}")

DIA = df.groupby("dia")[num].mean()
DIA.index = pd.to_datetime(DIA.index)
Cd = DIA.corr().abs()
A = Cd.to_numpy(copy=True); np.fill_diagonal(A, 1.0); A = np.nan_to_num(A, nan=0.0)
D = 1 - A; D = (D + D.T) / 2; np.fill_diagonal(D, 0.0)
Z = linkage(squareform(D, checks=False), method="average")

fig, ax = plt.subplots(figsize=(10, max(8, 0.2 * len(num))))
dendrogram(Z, labels=list(Cd.columns), orientation="left", leaf_font_size=7,
           color_threshold=0.10, ax=ax)
ax.axvline(0.10, color=PALETA["alerta"], ls="--", lw=1.2, label="|r| = 0,90")
ax.set_xlabel("distancia  1 − |r|")
ax.set_title(f"Agrupamiento jerárquico de las {len(num)} columnas (marco diario)")
ax.legend(frameon=False, loc="lower right"); ax.grid(False, axis="y")
plt.tight_layout()
guardar(fig, 26, "dendrograma",
        "Agrupamiento jerárquico de las columnas por similitud. Los grupos que se cierran por "
        "debajo de |r| = 0,90 contienen variantes de la misma información.")

## Figura 27 · PCA: varianza explicada

Cuántas **dimensiones independientes** hay realmente. Setenta y tantas columnas no son setenta y
tantas piezas de información.

Es diagnóstico, no propuesta de features: una componente mezcla previsiones con generación real
y pierde la etiqueta de fuga, que es lo último que conviene perder aquí.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Sólo columnas con buena cobertura: exigir las 77 a la vez deja pocos días y sesgados.
buenas = [c for c in num if DIA[c].notna().mean() > 0.95]
X = DIA[buenas].dropna()
print(f"PCA sobre {X.shape[1]} columnas y {len(X):,} días")

pca = PCA().fit(StandardScaler().fit_transform(X))
var = pca.explained_variance_ratio_ * 100
acum = var.cumsum()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].bar(range(1, min(21, len(var) + 1)), var[:20], color=PALETA["secundario"])
axes[0].set_xlabel("componente"); axes[0].set_ylabel("% de varianza")
axes[0].set_title("Varianza por componente (primeras 20)")
axes[1].plot(range(1, len(acum) + 1), acum, marker="o", ms=3, color=PALETA["principal"])
for u, col in [(90, PALETA["alerta"]), (95, PALETA["acento"])]:
    axes[1].axhline(u, color=col, ls="--", lw=1, label=f"{u}%")
    k = int((acum < u).sum() + 1)
    axes[1].annotate(f"{k} comp.", (k, u), fontsize=8, xytext=(4, -12),
                     textcoords="offset points", color=col)
axes[1].set_xlabel("nº de componentes"); axes[1].set_ylabel("% acumulado")
axes[1].set_title("Varianza acumulada"); axes[1].legend(frameon=False)
plt.tight_layout()
k90 = int((acum < 90).sum() + 1)
guardar(fig, 27, "pca_varianza",
        f"Varianza explicada por componente. Hacen falta {k90} componentes para el 90 % de la "
        f"varianza de {X.shape[1]} columnas: la dimensionalidad efectiva es mucho menor que el "
        f"número de variables.")

## Figura 28 · Cargas de las dos primeras componentes

El biplot de cargas: qué mide cada componente. Si una carga a la vez sobre columnas sin fuga y
con fuga, ilustra por qué **no sirve como feature** aunque explique mucha varianza.

In [ ]:
cargas = pd.DataFrame(pca.components_[:2].T, index=X.columns, columns=["PC1", "PC2"])
destacadas = cargas.abs().sum(axis=1).nlargest(18).index

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(cargas["PC1"], cargas["PC2"], s=14, color=PALETA["suave"])
for v in destacadas:
    x, y = cargas.loc[v, "PC1"], cargas.loc[v, "PC2"]
    ax.annotate(v[:26], (x, y), fontsize=6.5, xytext=(3, 3), textcoords="offset points")
    ax.plot([0, x], [0, y], lw=0.7, color=PALETA["secundario"], alpha=0.6)
ax.axhline(0, color="black", lw=0.8); ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel(f"PC1 ({var[0]:.1f}% de la varianza)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
ax.set_title("Cargas de las dos primeras componentes principales")
plt.tight_layout()
guardar(fig, 28, "pca_cargas",
        "Peso de cada variable en las dos primeras componentes. Variables próximas entre sí "
        "aportan información parecida; una componente que mezcla familias distintas no es "
        "utilizable como feature porque pierde la etiqueta de fuga.")

## Figura 29 · V de Cramér frente a Pearson

Réplica de `graficoVcramer` de las prácticas, con una vuelta de tuerca: se compara con la
correlación de Pearson. El V de Cramér **no supone linealidad**, así que una variable con V alto
y Pearson bajo tiene relación no lineal con el precio — y un modelo lineal la desaprovecharía.

In [ ]:
from scipy.stats import chi2_contingency

def v_cramer(v, target, q=5):
    d = pd.DataFrame({"v": v, "t": target}).dropna()
    if len(d) < 200:
        return np.nan
    cv = sorted(set(d["v"].quantile(np.linspace(0, 1, q + 1))))
    ct = sorted(set(d["t"].quantile(np.linspace(0, 1, q + 1))))
    if len(cv) < 3 or len(ct) < 3:
        return np.nan
    tabla = pd.crosstab(pd.cut(d["v"], bins=cv, include_lowest=True),
                        pd.cut(d["t"], bins=ct, include_lowest=True))
    chi2 = chi2_contingency(tabla)[0]
    return float(np.sqrt(chi2 / (tabla.sum().sum() * (min(tabla.shape) - 1))))

filas = []
for k, (col, fuga) in CAND.items():
    if col == PRECIO:
        continue
    filas.append({"variable": k, "fuga": fuga,
                  "Cramér": v_cramer(df[col], df[PRECIO]),
                  "Pearson": abs(df[col].corr(df[PRECIO]))})
V = pd.DataFrame(filas).dropna().set_index("variable").sort_values("Cramér")

fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(V))
ax.barh(y - 0.2, V["Cramér"], height=0.4, label="V de Cramér",
        color=[COLOR_FUGA.get(f, PALETA["neutro"]) for f in V["fuga"]])
ax.barh(y + 0.2, V["Pearson"], height=0.4, label="|Pearson|",
        color=PALETA["suave"], edgecolor="grey", linewidth=0.5)
ax.set_yticks(y, V.index)
ax.set_xlabel("asociación con el precio")
ax.set_title("Asociación de cualquier forma (Cramér) frente a lineal (Pearson)")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
guardar(fig, 29, "cramer_vs_pearson",
        "V de Cramér frente a correlación de Pearson. Una barra de color mucho más larga que "
        "la gris señala una relación no lineal, que un modelo lineal desaprovecharía.")

---
# Grupo C · Distribuciones

## Figura 30 · Histogramas de todas las candidatas

La rejilla de histogramas es el primer vistazo de cualquier EDA: enseña de golpe la forma, la
escala y las anomalías de cada variable. El color del marco recuerda la etiqueta de fuga.

In [ ]:
vs = [(k, v[0], v[1]) for k, v in CAND.items()]
ncol = 4
nfil = int(np.ceil(len(vs) / ncol))
fig, axes = plt.subplots(nfil, ncol, figsize=(3.3 * ncol, 2.3 * nfil))
for ax, (k, col, fuga) in zip(np.atleast_1d(axes).ravel(), vs):
    ax.hist(df[col].dropna(), bins=50, color=PALETA["secundario"])
    ax.set_title(k, fontsize=8.5, color=COLOR_FUGA.get(fuga, "black"))
    ax.tick_params(labelsize=6.5)
    for lado in ("bottom", "left"):
        ax.spines[lado].set_color(COLOR_FUGA.get(fuga, "black"))
        ax.spines[lado].set_linewidth(1.4)
for ax in np.atleast_1d(axes).ravel()[len(vs):]:
    ax.axis("off")
plt.tight_layout()
guardar(fig, 30, "histogramas_candidatas",
        "Distribución de cada variable candidata. El color del título y de los ejes indica su "
        "disponibilidad a las 12:00 de D.")

## Figura 31 · Diagramas de caja por familia

Cajas agrupadas por escala. Un boxplot de todo junto no serviría: la demanda va en decenas de
miles de MW y la temperatura en unidades. Separadas por familia, cada panel es legible.

In [ ]:
FAMILIAS = {
    "Demanda (MW)": ["demanda prevista", "demanda real"],
    "Renovable (MW)": ["eólica prevista", "eólica real", "solar prevista", "solar FV real"],
    "Despachable (MW)": ["hidráulica", "ciclo combinado"],
    "Precio y commodities": ["precio ES", "gas", "CO2"],
}
FAMILIAS = {k: [c for c in v if c in CAND] for k, v in FAMILIAS.items()}
FAMILIAS = {k: v for k, v in FAMILIAS.items() if v}

fig, axes = plt.subplots(1, len(FAMILIAS), figsize=(3.6 * len(FAMILIAS), 3.8))
for ax, (fam, ks) in zip(np.atleast_1d(axes), FAMILIAS.items()):
    datos = [df[CAND[k][0]].dropna() for k in ks]
    bp = ax.boxplot(datos, tick_labels=ks, showfliers=False, patch_artist=True)
    for parche, k in zip(bp["boxes"], ks):
        parche.set_facecolor(COLOR_FUGA.get(CAND[k][1], PALETA["neutro"]))
        parche.set_alpha(0.75)
    ax.set_title(fam, fontsize=9)
    ax.tick_params(axis="x", labelrotation=30, labelsize=7.5)
plt.tight_layout()
guardar(fig, 31, "cajas_por_familia",
        "Diagramas de caja agrupados por escala. El relleno indica la etiqueta de fuga: verde "
        "utilizable, rojo con fuga, ámbar con desfase.")

## Figura 32 · Q-Q del precio

El gráfico cuantil-cuantil contra la normal. Es la forma canónica de enseñar que **el precio no
es normal**: si los puntos se separan de la diagonal en los extremos, las colas son más pesadas
que las de una normal, y eso condiciona qué métricas y qué modelos tienen sentido.

Se compara con el logaritmo para valorar si una transformación arregla algo. Ojo: con precios
negativos el logaritmo no está definido, así que se aplica sobre un desplazamiento.

In [ ]:
from scipy import stats

p = df[PRECIO].dropna()
desp = p - p.min() + 1.0          # el log necesita valores positivos
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (serie, nombre) in zip(axes, [(p, "precio"), (np.log(desp), "log(precio desplazado)")]):
    stats.probplot(serie, dist="norm", plot=ax)
    ax.get_lines()[0].set(color=PALETA["secundario"], markersize=2)
    ax.get_lines()[1].set(color=PALETA["alerta"], lw=1.2)
    ax.set_title(f"Q-Q normal · {nombre}", fontsize=9)
    ax.set_xlabel("cuantiles teóricos"); ax.set_ylabel("cuantiles observados")
plt.tight_layout()
guardar(fig, 32, "qq_precio",
        f"Gráfico cuantil-cuantil del precio contra la normal (asimetría {p.skew():.2f}). La "
        f"separación de la diagonal en los extremos confirma colas más pesadas que las de una "
        f"normal, lo que condiciona la elección de métrica.")

## Figura 33 · Curva de duración de precios

Los precios ordenados de mayor a menor, un año por curva. Es la representación estándar del
sector eléctrico —la *price duration curve*— y responde de un vistazo: **cuántas horas al año
superan un precio dado**.

La parte plana central es el régimen habitual; los extremos son lo que decide el arbitraje.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
cmap = plt.get_cmap(CMAP_SEQ)
anios_ = sorted(df["anio"].unique())
for i, a in enumerate(anios_):
    s = df.loc[df["anio"] == a, PRECIO].dropna().sort_values(ascending=False).values
    ax.plot(np.linspace(0, 100, len(s)), s, lw=1.4,
            color=cmap(i / max(len(anios_) - 1, 1)), label=str(a))
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("% de horas del año por encima de ese precio")
ax.set_ylabel("EUR/MWh")
ax.set_title("Curva de duración de precios, por año")
ax.legend(frameon=False, ncol=2, title="año")
plt.tight_layout()
guardar(fig, 33, "curva_duracion_precios",
        "Curva de duración: precios ordenados de mayor a menor. Los extremos de cada curva son "
        "las horas que deciden el arbitraje; la parte central, el régimen habitual.")

---
# Grupo D · Estructura temporal

## Figura 34 · ACF y PACF

Las dos funciones canónicas del análisis de series temporales. La ACF mide la correlación total
con cada retardo; la PACF, la que queda **después de descontar los retardos intermedios**.

Leídas juntas dicen qué estructura autorregresiva tiene la serie, que es la información que
necesita un ARIMA y la que fija el listón para cualquier otro modelo.

In [ ]:
try:
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    d_ = df.groupby("dia")[PRECIO].mean()
    d_.index = pd.to_datetime(d_.index)
    d_ = d_.asfreq("D").interpolate(limit=3)

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    plot_acf(d_.dropna(), lags=40, ax=axes[0])
    axes[0].set_title("ACF del precio medio diario")
    plot_pacf(d_.dropna(), lags=40, ax=axes[1], method="ywm")
    axes[1].set_title("PACF del precio medio diario")
    for ax in axes:
        ax.set_xlabel("retardo (días)")
    plt.tight_layout()
    guardar(fig, 34, "acf_pacf",
            "Funciones de autocorrelación total y parcial del precio diario. El pico semanal y "
            "el decaimiento lento indican estructura autorregresiva con componente de calendario.")
    plt.show()
except ImportError:
    print("statsmodels no disponible: figura 34 omitida")

## Figura 35 · Descomposición estacional

Separa la serie en tendencia, estacionalidad y residuo. Es el gráfico que justifica cualquier
tratamiento de estacionalidad posterior: si la componente estacional es grande y estable,
modelarla explícitamente ahorra trabajo al modelo.

**El residuo es la parte interesante**: es lo que queda por explicar una vez descontados el
nivel y el ciclo, y es donde tienen que aportar las features exógenas.

In [ ]:
try:
    from statsmodels.tsa.seasonal import STL
    serie = df.groupby("dia")[PRECIO].mean()
    serie.index = pd.to_datetime(serie.index)
    serie = serie.asfreq("D").interpolate(limit=5).dropna()

    res = STL(serie, period=7, robust=True).fit()
    fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
    for ax, (s, nombre) in zip(axes, [(res.observed, "observado"), (res.trend, "tendencia"),
                                      (res.seasonal, "estacionalidad semanal"),
                                      (res.resid, "residuo")]):
        ax.plot(s.index, s.values, lw=0.7, color=PALETA["principal"])
        ax.set_ylabel(nombre, fontsize=8)
    axes[-1].axhline(0, color=PALETA["alerta"], lw=0.8)
    axes[0].set_title("Descomposición STL del precio medio diario")
    plt.tight_layout()
    var_resid = res.resid.var() / res.observed.var() * 100
    guardar(fig, 35, "descomposicion_stl",
            f"Descomposición en tendencia, estacionalidad semanal y residuo. El residuo "
            f"concentra el {var_resid:.0f} % de la varianza: es la parte que las features "
            f"exógenas tienen que explicar.")
    plt.show()
except ImportError:
    print("statsmodels no disponible: figura 35 omitida")

## Figura 36 · Densidades del precio por régimen

Las tres distribuciones superpuestas. Enseña que los regímenes no se distinguen solo por el
nivel medio sino por **la forma entera**: dispersión, asimetría y colas.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colores_reg = {"normal": PALETA["principal"], "excepción ibérica": PALETA["acento"],
               "apagón": PALETA["alerta"]}
for reg, sub in df.groupby("regimen"):
    s = sub[PRECIO].dropna()
    if len(s) < 100:
        continue
    s.plot(kind="density", ax=ax, lw=1.6, color=colores_reg.get(reg, PALETA["neutro"]),
           label=f"{reg} (n={len(s):,}, media {s.mean():.0f})")
ax.set_xlim(-30, min(400, df[PRECIO].quantile(0.999)))
ax.set_xlabel("EUR/MWh"); ax.set_ylabel("densidad")
ax.set_title("Distribución del precio según el régimen")
ax.legend(frameon=False)
plt.tight_layout()
guardar(fig, 36, "densidades_regimen",
        "Densidad del precio en cada régimen. No difieren sólo en nivel: cambian la dispersión "
        "y la forma, que es el argumento para tratarlos por separado y no promediarlos.")

## Figura 37 · Violines del precio por hora

El diagrama de violín añade sobre la caja la **forma completa** de la distribución. Aquí se ve
que las horas centrales no solo tienen media más baja: tienen distribución bimodal, con una masa
en precios muy bajos que no existe en las horas de tarde.

In [ ]:
ult = df[(df["anio"] == df["anio"].max()) & (df["regimen"] == "normal")]
if len(ult) < 1000:
    ult = df[df["regimen"] == "normal"]
datos = [ult.loc[ult["hora"] == h, PRECIO].dropna().values for h in range(24)]
datos = [d if len(d) > 10 else np.array([np.nan]) for d in datos]

fig, ax = plt.subplots(figsize=(11, 4))
vp = ax.violinplot(datos, positions=range(24), widths=0.85, showmeans=True, showextrema=False)
for cuerpo in vp["bodies"]:
    cuerpo.set_facecolor(PALETA["secundario"]); cuerpo.set_alpha(0.7)
vp["cmeans"].set_color(PALETA["alerta"])
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("Hora local"); ax.set_ylabel("EUR/MWh")
ax.set_title(f"Distribución del precio por hora del día ({ult['anio'].max()})")
plt.tight_layout()
guardar(fig, 37, "violines_hora",
        "Distribución completa del precio en cada hora. Las horas centrales muestran una masa "
        "en precios muy bajos que la media por sí sola no revela.")

---
# Grupo E · Estabilidad y evaluación

## Figura 38 · Correlación con el precio, año a año

Un mapa de calor con las candidatas en filas y los años en columnas. Responde a la pregunta que
una correlación global no puede: **¿es estable esta relación?**

Una fila que cambia de color entre años es una relación que no generaliza, y una feature elegida
por ella envejecería mal.

In [ ]:
filas = []
for k, (col, fuga) in CAND.items():
    if col == PRECIO:
        continue
    filas.append(df.groupby("anio").apply(
        lambda g: g[PRECIO].corr(g[col]), include_groups=False).rename(k))
E = pd.DataFrame(filas)

fig, ax = plt.subplots(figsize=(8, 0.42 * len(E) + 1.6))
im = ax.imshow(E.values, cmap=CMAP_DIV, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1),
               aspect="auto")
ax.set_xticks(range(E.shape[1]), E.columns)
ax.set_yticks(range(len(E)), E.index)
for et, k in zip(ax.get_yticklabels(), E.index):
    et.set_color(COLOR_FUGA.get(CAND[k][1], "black"))
for i in range(E.shape[0]):
    for j in range(E.shape[1]):
        v = E.iloc[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if abs(v) > 0.55 else "black")
ax.set_title("Correlación de cada driver con el precio, por año")
sin_rejilla(ax); fig.colorbar(im, ax=ax, shrink=0.8, label="r")
plt.tight_layout()
guardar(fig, 38, "correlacion_por_anio",
        "Estabilidad de cada relación a lo largo del tiempo. Una fila que cambia de signo o de "
        "intensidad señala una relación que no generaliza fuera de su período.")

## Figura 39 · Dispersión del precio frente a los tres drivers sin fuga

Los diagramas de dispersión con la nube coloreada por hora del día. Es la vista que revela
**relaciones no lineales y condicionadas**: la misma solar prevista significa cosas distintas a
las 8 de la mañana y a las 2 de la tarde.

In [ ]:
sin_fuga = [k for k, (c, f) in CAND.items() if f == "SIN FUGA"][:3]
if sin_fuga:
    fig, axes = plt.subplots(1, len(sin_fuga), figsize=(4.3 * len(sin_fuga), 3.8))
    for ax, k in zip(np.atleast_1d(axes), sin_fuga):
        col = CAND[k][0]
        m = df[[col, PRECIO, "hora"]].dropna().sample(min(12000, len(df)), random_state=0)
        sc = ax.scatter(m[col], m[PRECIO], c=m["hora"], cmap="twilight", s=3, alpha=0.5)
        ax.set_xlabel(k); ax.set_ylabel("EUR/MWh" if k == sin_fuga[0] else "")
        ax.set_title(f"precio vs {k}", fontsize=9)
    fig.colorbar(sc, ax=axes, shrink=0.85, label="hora local")
    guardar(fig, 39, "dispersion_sin_fuga",
            "Precio frente a los drivers utilizables, coloreado por hora del día. El color "
            "revela que la misma variable significa cosas distintas según la hora: la relación "
            "es condicionada, no simple.")
    plt.show()

## Figura 40 · Coste de la confusión

El mapa de calor de la matriz de coste: cuántos euros por MWh se pierden al confundir una clase
de hora con otra. **No todos los errores cuestan lo mismo**, y una métrica simétrica como el MAE
no lo recoge.

Es la traducción del EDA a la unidad en que se mide el proyecto, y el argumento de por qué
`F11_baselines` evalúa también por dinero capturado.

In [ ]:
K = 4
d = df[["dia", "hora", PRECIO]].dropna(subset=[PRECIO]).copy()
d["rango"] = d.groupby("dia")[PRECIO].rank(method="first")
d["n"] = d.groupby("dia")[PRECIO].transform("size")
d["clase"] = "intermedia"
d.loc[d["rango"] <= K, "clase"] = "barata"
d.loc[d["rango"] > d["n"] - K, "clase"] = "cara"

medias = d.groupby("clase")[PRECIO].mean()
clases = ["barata", "intermedia", "cara"]
COSTE = pd.DataFrame([[abs(medias[a] - medias[b]) for b in clases] for a in clases],
                     index=clases, columns=clases)

fig, ax = plt.subplots(figsize=(5.4, 4.4))
im = ax.imshow(COSTE.values, cmap="Reds")
ax.set_xticks(range(3), [f"predicho\n{c}" for c in clases])
ax.set_yticks(range(3), [f"real {c}" for c in clases])
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{COSTE.iloc[i, j]:.1f}", ha="center", va="center", fontsize=11,
                color="white" if COSTE.iloc[i, j] > COSTE.values.max() * 0.6 else "black")
ax.set_title("Coste de confundir una clase de hora con otra\n(EUR/MWh)")
sin_rejilla(ax); fig.colorbar(im, ax=ax, shrink=0.8, label="EUR/MWh")
plt.tight_layout()
peor = COSTE.loc["cara", "barata"]
medio = COSTE.loc["cara", "intermedia"]
guardar(fig, 40, "coste_confusion",
        f"Coste de cada tipo de error. Confundir una hora cara con una barata cuesta "
        f"{peor:.1f} EUR/MWh, {peor/max(medio, 1e-9):.1f} veces más que confundirla con una "
        f"intermedia: la métrica de evaluación no puede ser simétrica.")

---
# Láminas compuestas

Cinco composiciones de cuatro paneles. Cada una ocupa **media página** y cuenta una historia
completa, en vez de gastar una página por gráfico.

Es la forma práctica de enseñar mucho análisis en poco espacio: el pie lee la lámina como
conjunto, no panel a panel.

In [ ]:
def lamina(nombre, titulo, pie, paneles, ncol=2, alto=3.2, num=None):
    '''
    Compone varios paneles en una figura. Cada panel es una función que recibe un eje.
    Si un panel falla, se marca en el propio hueco en vez de tumbar la lámina entera:
    es preferible una lámina con tres paneles y un aviso que ninguna lámina.
    '''
    nfil = int(np.ceil(len(paneles) / ncol))
    fig, axes = plt.subplots(nfil, ncol, figsize=(6.2 * ncol, alto * nfil))
    ejes = np.atleast_1d(axes).ravel()
    for ax, (subtit, fn) in zip(ejes, paneles):
        try:
            fn(ax)
            ax.set_title(subtit, fontsize=9)
        except Exception as e:
            ax.axis("off")
            ax.text(0.5, 0.5, f"{subtit}\nno disponible\n({type(e).__name__})",
                    ha="center", va="center", fontsize=8, color=PALETA["neutro"])
    for ax in ejes[len(paneles):]:
        ax.axis("off")
    fig.suptitle(titulo, fontsize=11, y=1.005)
    plt.tight_layout()
    guardar(fig, num or nombre, nombre, pie)
    plt.show()

## Lámina 1 · El target de un vistazo

In [ ]:
def _p_serie(ax):
    s = df.groupby("dia")[PRECIO].mean()
    ax.plot(pd.to_datetime(s.index), s.values, lw=0.6, color=PALETA["principal"])
    ax.set_ylabel("EUR/MWh")

def _p_hist(ax):
    ax.hist(df[PRECIO].dropna(), bins=90, color=PALETA["secundario"])
    ax.set_yscale("log"); ax.set_xlabel("EUR/MWh")

def _p_perfil(ax):
    pv = df[df["regimen"] == "normal"].pivot_table(index="hora", columns="anio",
                                                   values=PRECIO, aggfunc="mean")
    pv.plot(ax=ax, lw=1.1, colormap=CMAP_SEQ, legend=False)
    ax.set_xlabel("hora local"); ax.set_ylabel("EUR/MWh"); ax.set_xticks(range(0, 24, 4))

def _p_dur(ax):
    for i, a in enumerate(sorted(df["anio"].unique())):
        s = df.loc[df["anio"] == a, PRECIO].dropna().sort_values(ascending=False).values
        ax.plot(np.linspace(0, 100, len(s)), s, lw=1.1,
                color=plt.get_cmap(CMAP_SEQ)(i / max(df["anio"].nunique() - 1, 1)))
    ax.set_xlabel("% de horas"); ax.set_ylabel("EUR/MWh")

lamina("lamina1_target", "El target: nivel, distribución, forma horaria y colas",
       "Cuatro vistas del precio: evolución diaria, distribución en escala logarítmica, perfil "
       "horario por año y curva de duración. En conjunto muestran que el problema cambió de "
       "naturaleza durante el propio período de entrenamiento.",
       [("Precio medio diario", _p_serie), ("Distribución (log)", _p_hist),
        ("Perfil horario por año", _p_perfil), ("Curva de duración", _p_dur)],
       num="L1")

## Lámina 2 · Estacionalidades

In [ ]:
def _p_mh(ax):
    h = df[df["regimen"] == "normal"].pivot_table(index="hora", columns="mes",
                                                  values=PRECIO, aggfunc="mean")
    im = ax.imshow(h.values, aspect="auto", origin="lower", cmap=CMAP_SEQ)
    ax.set_xticks(range(12), list("EFMAMJJASOND")); ax.set_ylabel("hora"); sin_rejilla(ax)

def _p_dh(ax):
    h = df[df["regimen"] == "normal"].pivot_table(index="hora", columns="dow",
                                                  values=PRECIO, aggfunc="mean")
    im = ax.imshow(h.values, aspect="auto", origin="lower", cmap=CMAP_SEQ)
    ax.set_xticks(range(7), ["L", "M", "X", "J", "V", "S", "D"]); sin_rejilla(ax)

def _p_mes(ax):
    df.groupby("mes")[PRECIO].mean().plot(ax=ax, marker="o", color=PALETA["principal"])
    ax.set_xticks(range(1, 13)); ax.set_xlabel("mes"); ax.set_ylabel("EUR/MWh")

def _p_dow(ax):
    df.groupby("dow")[PRECIO].mean().plot(kind="bar", ax=ax, color=PALETA["secundario"])
    ax.set_xticks(range(7), ["L", "M", "X", "J", "V", "S", "D"], rotation=0)
    ax.set_xlabel(""); ax.set_ylabel("EUR/MWh")

lamina("lamina2_estacionalidad", "Las tres estacionalidades del precio: anual, semanal y diaria",
       "Mapas de calor mes×hora y día×hora, con sus perfiles marginales. La interacción entre "
       "ciclos es lo que obliga a que el modelo trate las 24 horas por separado.",
       [("Mes × hora", _p_mh), ("Día de la semana × hora", _p_dh),
        ("Perfil mensual", _p_mes), ("Perfil semanal", _p_dow)], num="L2")

## Lámina 3 · Estructura entre variables

In [ ]:
def _p_corr(ax):
    m = np.triu(np.ones_like(Cm, dtype=bool))
    ax.imshow(np.where(m, np.nan, Cm.values), cmap=CMAP_DIV,
              norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1))
    ax.set_xticks(range(len(Cm)), Cm.columns, rotation=90, fontsize=6)
    ax.set_yticks(range(len(Cm)), Cm.columns, fontsize=6); sin_rejilla(ax)

def _p_scree(ax):
    ax.bar(range(1, min(16, len(var) + 1)), var[:15], color=PALETA["secundario"])
    ax.set_xlabel("componente"); ax.set_ylabel("% varianza")

def _p_cramer(ax):
    V.sort_values("Cramér").plot(kind="barh", y="Cramér", ax=ax, legend=False,
                                 color=[COLOR_FUGA.get(f, PALETA["neutro"])
                                        for f in V.sort_values("Cramér")["fuga"]])
    ax.set_xlabel("V de Cramér"); ax.tick_params(labelsize=7)

def _p_anio(ax):
    ax.imshow(E.values, cmap=CMAP_DIV, norm=TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1),
              aspect="auto")
    ax.set_xticks(range(E.shape[1]), E.columns, fontsize=7)
    ax.set_yticks(range(len(E)), E.index, fontsize=6.5); sin_rejilla(ax)

lamina("lamina3_estructura", "Estructura entre variables: redundancia, dimensión y estabilidad",
       "Matriz de correlación, varianza por componente principal, asociación no lineal y "
       "estabilidad año a año. Juntas justifican qué variables sobran y cuáles no generalizan.",
       [("Correlación entre candidatas", _p_corr), ("Varianza por componente", _p_scree),
        ("V de Cramér con el precio", _p_cramer), ("Correlación por año", _p_anio)], num="L3")

## Lámina 4 · Calidad de los datos

In [ ]:
def _p_nulos(ax):
    n = df.isna().mean().mul(100).sort_values(ascending=False).head(20)
    ax.barh(n.index[::-1], n.values[::-1], color=PALETA["neutro"])
    ax.set_xlabel("% de nulos"); ax.tick_params(labelsize=6)

def _p_ausencias(ax):
    cs = [c for c in df.columns if c.startswith("era5_")]
    if len(cs) < 2:
        raise ValueError("sin era5")
    ax.imshow(df[cs].isna().corr().values, cmap=CMAP_DIV, vmin=-1, vmax=1)
    ax.set_xticks(range(len(cs)), [c[5:20] for c in cs], rotation=90, fontsize=6)
    ax.set_yticks(range(len(cs)), [c[5:20] for c in cs], fontsize=6); sin_rejilla(ax)

def _p_cobertura(ax):
    cob = df.set_index("ts_utc")[[PRECIO]].notna().resample("MS").mean().mul(100)
    ax.plot(cob.index, cob.values, lw=1.2, color=PALETA["ok"])
    ax.set_ylim(0, 105); ax.set_ylabel("% horas con precio")

def _p_dst(ax):
    hd = df.groupby("dia").size()
    at = hd[hd != 24]
    ax.bar(range(len(at)), at.values, color=PALETA["acento"])
    ax.axhline(24, color=PALETA["alerta"], ls="--", lw=1)
    ax.set_xticks(range(len(at)), [str(i) for i in at.index], rotation=90, fontsize=6)
    ax.set_ylabel("horas del día"); ax.set_ylim(22, 26)

lamina("lamina4_calidad", "Calidad de los datos: nulos, patrón de ausencias, cobertura y DST",
       "Los nulos de las tablas diarias y de ERA5 son estructurales, no huecos. Los días con 23 "
       "y 25 horas corresponden exactamente a los cambios de horario.",
       [("Columnas con más nulos", _p_nulos), ("Patrón de ausencias en ERA5", _p_ausencias),
        ("Cobertura mensual del target", _p_cobertura), ("Días con ≠ 24 horas", _p_dst)],
       num="L4")

## Lámina 5 · Del precio a la decisión de la batería

In [ ]:
def _p_clases(ax):
    tab = pd.crosstab(d["hora"], d["clase"], normalize="index") * 100
    ax.plot(tab.index, tab["cara"], marker="o", ms=3, color=PALETA["alerta"], label="cara")
    ax.plot(tab.index, tab["barata"], marker="o", ms=3, color=PALETA["secundario"],
            label="barata")
    ax.axhline(100 * K / 24, color=PALETA["neutro"], ls="--", lw=1)
    ax.set_xlabel("hora local"); ax.set_ylabel("% de días"); ax.legend(frameon=False, fontsize=7)

def _p_coste(ax):
    ax.imshow(COSTE.values, cmap="Reds")
    ax.set_xticks(range(3), clases, fontsize=7); ax.set_yticks(range(3), clases, fontsize=7)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{COSTE.iloc[i, j]:.0f}", ha="center", va="center", fontsize=9)
    sin_rejilla(ax)

def _p_spread(ax):
    sp = d.groupby("dia").apply(
        lambda g: g.nlargest(K, PRECIO)[PRECIO].mean() - g.nsmallest(K, PRECIO)[PRECIO].mean(),
        include_groups=False)
    sp.index = pd.to_datetime(sp.index)
    sp.resample("MS").mean().plot(ax=ax, color=PALETA["principal"], lw=1.3)
    ax.set_ylabel("EUR/MWh"); ax.set_xlabel("")

def _p_medias(ax):
    medias.reindex(clases).plot(kind="bar", ax=ax,
                                color=[PALETA["secundario"], PALETA["neutro"], PALETA["alerta"]])
    ax.set_xticks(range(3), clases, rotation=0); ax.set_ylabel("EUR/MWh"); ax.set_xlabel("")

lamina("lamina5_bess", "Del precio a la decisión de operar la batería",
       "Cuándo caen las horas extremas, cuánto separa el precio a cada clase, cómo evoluciona "
       "el diferencial aprovechable y cuánto cuesta cada tipo de error. Es el puente entre el "
       "EDA del precio y la optimización del almacenamiento.",
       [("Probabilidad de hora extrema", _p_clases), ("Coste de la confusión", _p_coste),
        ("Diferencial diario aprovechable", _p_spread), ("Precio medio por clase", _p_medias)],
       num="L5")

---
## Cierre · pies de figura del anexo

In [ ]:
ruta = SALIDA / "pies_de_figura_anexo.md"
with ruta.open("w", encoding="utf-8") as f:
    f.write("# Pies de figura · anexo (21-40 y láminas)\n\n")
    f.write("*Generado por `11_figuras_anexo.ipynb`. No editar a mano: se regenera.*\n\n")
    for etq, fichero, pie in PIES:
        f.write(f"**{etq}** — `{fichero}`\n\n{pie}\n\n")
print(f"{len(PIES)} figuras · {ruta}")